# portfolio_D_deploy — 分散案D[RG3] 運用カード(審査/資金化後収益/高リスク)

推奨構成 **v4:v7:E-Mon[RG3]:E5 = 30:25:25:20**(docs/75 §6 最小失格)の運用数値を一括算出(docs/76)。
Drive優先(指数=docs/68同一入力)。①→②(任意)→③実行。実行15分前後。
規律: 審査=median-3は口座喪失前提の攻め・資金化後は生存優先。¥はYahoo近似(Driveで確定)・月次=楽観側。


In [ ]:
# ① 依存ピン留め(再現性) — 基準環境は Colab 実測の numpy 2.0.2 / pandas 2.2.2(2026-06 検証実行と同一)。
# 現行 Colab なら通常 no-op。バージョンが変わった旨が表示されたらランタイム再起動してから先へ。
# ※ numpy 1.x へのダウングレードは pandas wheel と ABI 非互換(dtype size changed)になるため不可。
!pip install -q numpy==2.0.2 pandas==2.2.2 matplotlib==3.10.0


## ② save_result(任意・本体末尾の証跡保存に使用)

In [ ]:
# -*- coding: utf-8 -*-
"""検証結果と入力データのハッシュを research/results/ に保存(再現性の証跡)。

docs/71 §4 の残課題対応。Dukascopy 実データで検証ノートを回す際、看板数値を docs に
"手転記"する代わりに、本ヘルパーで metrics + 入力SHA-256 + 実行環境バージョンを JSON 化する。
これにより「どの入力・どの環境で出た数字か」がリポジトリ内で追跡可能になる。

使い方(Colab/ローカル共通):
    from capture_results import save_result
    save_result(
        "v7_10year_validation",
        metrics={"net_pct": 80.3, "maxDD_pct": 14.8, "phase1_pass_pct": 79.2},
        inputs=[f"{H1_DIR}/EURJPY_h1.csv", f"{H1_DIR}/GBPJPY_h1.csv"],
        params=P, seed=MC_SEED,
    )
    # -> research/results/v7_10year_validation.json
"""
import os, sys, json, hashlib, platform
import datetime as _dtmod        # エイリアスで保持(ノートで `from datetime import datetime` に上書きされても壊れない)

try:
    _BASE = os.path.dirname(os.path.abspath(__file__))
except NameError:            # ノート/Colab では __file__ が無い
    _BASE = os.getcwd()
RESULTS_DIR = os.path.join(_BASE, "results")


def sha256_file(path, _buf=1 << 20):
    """ファイルの SHA-256(同一入力であることの証跡)。"""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(_buf), b""):
            h.update(chunk)
    return h.hexdigest()


def _env():
    out = {"python": sys.version.split()[0], "platform": platform.platform()}
    for mod in ("numpy", "pandas", "matplotlib"):
        try:
            out[mod] = getattr(__import__(mod), "__version__", "?")
        except Exception:
            out[mod] = None
    return out


def save_result(name, metrics, inputs=None, params=None, seed=None):
    """検証結果を research/results/<name>.json に保存して返す。

    name    : 出力ファイル名(拡張子不要)
    metrics : 看板数値の dict(例 {"net_pct":80.3,"maxDD_pct":14.8})
    inputs  : 入力データファイルのパス list(各 SHA-256 を記録)
    params  : EA入力など(再現に必要な設定)
    seed    : 乱数シード
    """
    inputs = inputs or []
    rec = {
        "name": name,
        "saved_at": _dtmod.datetime.now(_dtmod.timezone.utc).isoformat(),
        "env": _env(),
        "seed": seed,
        "params": params,
        "inputs": [
            {
                "file": os.path.basename(p),
                "exists": os.path.exists(p),
                "bytes": os.path.getsize(p) if os.path.exists(p) else None,
                "sha256": sha256_file(p) if os.path.exists(p) else None,
            }
            for p in inputs
        ],
        "metrics": metrics,
    }
    os.makedirs(RESULTS_DIR, exist_ok=True)
    out = os.path.join(RESULTS_DIR, f"{name}.json")
    with open(out, "w") as f:
        json.dump(rec, f, ensure_ascii=False, indent=2, default=str)
    missing = [i["file"] for i in rec["inputs"] if not i["exists"]]
    print(f"[capture] {out}  inputs={len(inputs)}" + (f"  ★未検出={missing}" if missing else ""))
    return out



## ③ 本体 — 取込→分散案D構築→審査(標準/高リスク)→資金化後収益

In [ ]:
# -*- coding: utf-8 -*-
"""median3_rg3_target — 中央3ヶ月の最良ポートフォリオを E-Mon raw vs RG3 で確定(自己完結・Drive優先)。
docs/68(RG3採用=median-4で確定)の median-3 版。中央値定義は median3_target.py と同一(通過パスのみ)。
Drive(multiasset_daily)があれば docs/68 と同一入力=確定値。無ければ固定窓Yahoo=相対比較(差分を読む)。
規律: 数字は盛らない。月次解像度=失格は楽観。最終はデモで確定。"""
import os, sys, json, shutil, urllib.request, csv as _csv, numpy as np, pandas as pd, warnings
import datetime as _dtm
warnings.filterwarnings("ignore")
HERE = os.getcwd(); DATA = os.path.join(HERE, "data"); os.makedirs(DATA, exist_ok=True)
P1, P2 = 1451606400, 1767225599      # 固定窓 2016-01-01..2025-12-31
YS = {**{f: f+"=X" for f in ["EURUSD","GBPUSD","USDJPY","AUDUSD","USDCHF","USDCAD","NZDUSD","EURJPY","GBPJPY"]},
      "NAS100":"%5EIXIC","US500":"%5EGSPC","GER40":"%5EGDAXI","XAUUSD":"GC=F"}
DRIVE_BASE = "/content/drive/MyDrive/forex_ml"; INDEXES = ["NAS100","US500","GER40"]
try:
    if not os.path.exists("/content/drive/MyDrive"):
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print("Drive mount注意(継続):", e)
drive_got = []
_src = os.path.join(DRIVE_BASE, "multiasset_daily")
if os.path.isdir(_src):
    for _nm in INDEXES:
        _s = os.path.join(_src, f"{_nm}_d.csv")
        if os.path.exists(_s):
            shutil.copy(_s, os.path.join(DATA, f"{_nm}_d.csv")); drive_got.append(_nm)
print("[データ] Drive取込=", drive_got or "なし→固定窓Yahoo")

def _fetch(name):
    p = os.path.join(DATA, f"{name}_d.csv")
    if os.path.exists(p): return
    u = f"https://query2.finance.yahoo.com/v8/finance/chart/{YS[name]}?interval=1d&period1={P1}&period2={P2}"
    d = json.loads(urllib.request.urlopen(urllib.request.Request(u, headers={"User-Agent":"Mozilla/5.0"}), timeout=25).read())
    r = d["chart"]["result"][0]; ts = r["timestamp"]; q = r["indicators"]["quote"][0]
    rows = [(_dtm.datetime.fromtimestamp(t, _dtm.timezone.utc).replace(tzinfo=None).strftime("%Y-%m-%d %H:%M:%S"),
             q["open"][i], q["high"][i], q["low"][i], q["close"][i])
            for i, t in enumerate(ts) if None not in (q["open"][i], q["high"][i], q["low"][i], q["close"][i])]
    with open(p, "w", newline="") as f:
        w = _csv.writer(f); w.writerow(["timestamp","open","high","low","close"]); w.writerows(rows)
for _nm in YS: _fetch(_nm)
print("data ready:", len([x for x in os.listdir(DATA) if x.endswith('_d.csv')]), "files")

# ====== 系列ビルダ(parallel_vs_existing_compare.py 逐語) ======
# 定数(parallel_vs_existing_compare.py 冒頭より逐語)
E5_BASKET = ["XAUUSD", "US500", "NAS100", "GER40"]
YEN = ["EURJPY", "GBPJPY", "USDJPY"]
V4_PAIRS = ["EURUSD", "GBPUSD", "USDJPY", "AUDUSD", "USDCHF", "USDCAD", "NZDUSD", "EURJPY", "GBPJPY"]

def load_daily(name, crypto=False):
    df = pd.read_csv(os.path.join(DATA, f"{name}_d.csv"))
    df["t"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df = df.dropna(subset=["t"]).sort_values("t")
    df["trade_date"] = (df["t"] + pd.Timedelta(hours=2)).dt.floor("D")
    df = df.groupby("trade_date", as_index=True).last()
    df["weekday"] = df.index.dayofweek
    if not crypto:
        df = df[df["weekday"] <= 4]
    df["o2o"] = df["open"].shift(-1) / df["open"] - 1.0
    return df


def to_monthly(r):
    s = r.copy(); s.index = pd.to_datetime(s.index).to_period("M"); return s.groupby(level=0).sum()


def pip_size(p): return 0.01 if p.endswith("JPY") else 0.0001
def v7_monthly():
    acc = None
    for p in YEN:
        df = load_daily(p); m = to_monthly((df[df["weekday"] == 0]["o2o"] - 2 * pip_size(p) / df[df["weekday"] == 0]["open"]).dropna())
        acc = m if acc is None else acc.add(m, fill_value=0.0)
    return (acc / 3.0).rename("v7")

def e5_monthly():
    closes = {}
    for nm in E5_BASKET:
        df = load_daily(nm); s = df["close"]; s.index = pd.to_datetime(df.index)
        closes[nm] = s.resample("ME").last()
    px = pd.DataFrame(closes).dropna(); ret = px.pct_change()
    sig = pd.DataFrame(0.0, index=px.index, columns=px.columns)
    for lb in (1, 3, 6, 12):
        sig = sig.add(np.sign(px.pct_change(lb)), fill_value=0)
    pos = np.sign(sig).shift(1)
    out = ((pos * ret).mean(axis=1) - 5e-4).dropna(); out.index = out.index.to_period("M")
    return out.rename("E5")

def rsi_wilder(c, n=14):
    d = np.diff(c, prepend=c[0]); up = np.clip(d, 0, None); dn = np.clip(-d, 0, None)
    au = np.empty_like(c); ad = np.empty_like(c); au[0] = up[0]; ad[0] = dn[0]; a = 1.0 / n
    for i in range(1, len(c)):
        au[i] = a * up[i] + (1 - a) * au[i - 1]; ad[i] = a * dn[i] + (1 - a) * ad[i - 1]
    rs = au / np.where(ad == 0, 1e-12, ad); return 100 - 100 / (1 + rs)


def atr_daily(h, l, c, n=14):
    pc = np.roll(c, 1); pc[0] = c[0]
    tr = np.maximum(h - l, np.maximum(np.abs(h - pc), np.abs(l - pc)))
    out = np.empty_like(tr); out[0] = tr[0]; a = 1.0 / n
    for i in range(1, len(tr)):
        out[i] = a * tr[i] + (1 - a) * out[i - 1]
    return out
def v4_monthly():
    """FX 9ペア 日足k≥4合議。SL=1.5ATR/RR1.2/8日時間切れを日足OHLCで簡易再現。月次合算。"""
    monthly = {}
    for p in V4_PAIRS:
        path = os.path.join(DATA, f"{p}_d.csv")
        if not os.path.exists(path):
            continue
        df = load_daily(p)
        o = df["open"].values; h = df["high"].values; l = df["low"].values; c = df["close"].values
        idx = df.index; n = len(c)
        rsi = rsi_wilder(c, 14); atr = atr_daily(h, l, c, 14)
        bbwin = 20; cost = 2 * pip_size(p)
        i = bbwin + 2
        while i < n - 1:
            # 直近確定足 i で4条件集計
            win = c[i - bbwin:i]; mean = win.mean(); sd = win.std(ddof=1)
            z = (c[i] - mean) / sd if sd > 0 else 0.0
            down = 0
            for k in range(0, 12):
                if i - k - 1 >= 0 and c[i - k] < c[i - k - 1]:
                    down += 1
                else:
                    break
            up = 0
            for k in range(0, 12):
                if i - k - 1 >= 0 and c[i - k] > c[i - k - 1]:
                    up += 1
                else:
                    break
            ret = (c[i] - c[i - 1]) / c[i - 1] if c[i - 1] else 0.0
            mv = 0.005
            buy = int(rsi[i] < 35) + int(z < -1.5) + int(down >= 3) + int(ret < -mv)
            sell = int(rsi[i] > 65) + int(z > 1.5) + int(up >= 3) + int(ret > mv)
            sig = 1 if (buy >= 4 and buy > sell) else (-1 if (sell >= 4 and sell > buy) else 0)
            if sig == 0:
                i += 1; continue
            # 翌足 i+1 open でエントリ
            entry = o[i + 1]; sld = 1.5 * atr[i]; tpd = 1.2 * sld
            if sld <= 0:
                i += 1; continue
            sl = entry - sig * sld; tp = entry + sig * tpd
            exit_px = None
            j = i + 1; held = 0
            while j < n and held < 8:
                hi = h[j]; lo = l[j]
                if sig > 0:
                    if lo <= sl:
                        exit_px = sl; break
                    if hi >= tp:
                        exit_px = tp; break
                else:
                    if hi >= sl:
                        exit_px = sl; break
                    if lo <= tp:
                        exit_px = tp; break
                j += 1; held += 1
            if exit_px is None:
                exit_px = c[min(j, n - 1)]
            r = sig * (exit_px / entry - 1.0) - cost / entry
            mkey = pd.Period(idx[min(j, n - 1)], freq="M")
            monthly[mkey] = monthly.get(mkey, 0.0) + r
            i = max(i + 1, j)   # 決済後の翌足から再評価
    s = pd.Series(monthly).sort_index()
    return s.rename("v4")
def z(s, target=0.01):
    sd = s.std()
    return s / sd * target if sd > 0 else s

# ====== E-Mon raw / RG3 (edge_regime_gate_10y.py のロジックを load_daily 上に逐語移植) ======
BASE_BPS = 5.0; VOL_WIN = 20; VOL_Q = 0.80; SMA_WIN = 200; EMON = INDEXES; WD_MON = 0
_C = {}
def load(name):
    if name not in _C: _C[name] = load_daily(name)
    return _C[name]

def realized_vol(close, win=VOL_WIN):
    r = np.log(close).diff()
    return (r.rolling(win).std() * np.sqrt(252)).shift(1)      # 前足までで確定

def sma_above(close, win=SMA_WIN):
    return (close > close.rolling(win).mean()).shift(1)        # 前足までで確定

def basket_vol_high(names):
    vols = [realized_vol(load(nm)["close"]).rename(nm) for nm in names]
    bv = pd.concat(vols, axis=1).mean(axis=1).dropna()
    thr = bv.rolling(252, min_periods=60).quantile(VOL_Q)
    return (bv > thr)

def emon_weekly_raw(cost_mult=1.0):
    legs = []
    for nm in EMON:
        df = load(nm); c = BASE_BPS * 1e-4 * cost_mult
        legs.append((df[df["weekday"] == WD_MON]["o2o"] - c).rename(nm))
    return pd.concat(legs, axis=1).mean(axis=1).dropna()

def basket_regime(names):
    hv_high = basket_vol_high(names)
    up_parts = [sma_above(load(nm)["close"]).rename(nm) for nm in names]
    up_maj = (pd.concat(up_parts, axis=1).astype(float).mean(axis=1) >= 0.5)
    return up_maj, hv_high

def emon_weekly_gated(kind="riskoff", cost_mult=1.0):
    raw = emon_weekly_raw(cost_mult)
    up_maj, hv_high = basket_regime(EMON)
    up = up_maj.reindex(raw.index).fillna(False).astype(bool)
    hv = hv_high.reindex(raw.index).fillna(False).astype(bool)
    on = ~((~up) & hv)                        # SMA割れ かつ 高ボラ の真リスクオフ週だけ見送り(非対称)
    return raw.where(on, np.nan).dropna()

# ====== MC(pstar_challenge_mc.py 逐語) ======
def block_paths(series, n_paths, horizon, block=3, seed=11):
    w = np.asarray(series, float); n = len(w)
    rng = np.random.default_rng(seed); P = np.empty((n_paths, horizon))
    for p in range(n_paths):
        seq = []
        while len(seq) < horizon:
            st = rng.integers(0, n)
            seq.extend(w[(st + k) % n] for k in range(block))
        P[p] = seq[:horizon]
    return P


def p95_annual_maxdd(series, scale, n_paths=4000):
    P = block_paths(series.values * scale, n_paths, 12, seed=3)
    dds = []
    for i in range(n_paths):
        eq = np.cumprod(1 + P[i]); dd = ((eq - np.maximum.accumulate(eq)) / np.maximum.accumulate(eq)).min()
        dds.append(dd)
    return float(np.percentile(dds, 5) * 100)


def find_scale(series, target_dd, lo=0.05, hi=6.0):
    for _ in range(40):
        mid = (lo + hi) / 2
        if p95_annual_maxdd(series, mid) <= target_dd:   # より悪い(負が大きい)
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2


def simulate(series, scale, target_gain, max_loss=0.10, n_paths=20000, cap_m=72, block=3, seed=7):
    """月次パスで +target_gain 到達(通過) vs −max_loss 抵触(失格)。通過月数分布を返す。"""
    P = block_paths(series.values * scale, n_paths, cap_m, block=block, seed=seed)
    months = []; passed = 0; failed = 0; timeout = 0
    for i in range(n_paths):
        eq = 1.0; done = False
        for m in range(cap_m):
            eq *= (1 + P[i, m])
            if eq <= 1.0 - max_loss:
                failed += 1; done = True; break
            if eq >= 1.0 + target_gain:
                passed += 1; months.append(m + 1); done = True; break
        if not done:
            timeout += 1
    months = np.array(months) if months else np.array([cap_m])
    return dict(pass_pct=round(passed / n_paths * 100, 1), fail_pct=round(failed / n_paths * 100, 1),
                timeout_pct=round(timeout / n_paths * 100, 1),
                med_month=int(np.median(months)), p25=int(np.percentile(months, 25)),
                p75=int(np.percentile(months, 75)), p90=int(np.percentile(months, 90)))

# ====== 分散案D[RG3] 運用カード: 審査(標準/高リスク) + 資金化後収益 ======
ACCOUNT_USD = 100000.0; SPLIT = 0.95; FX = 155.0
WEIGHTS = {"v4": .30, "v7": .25, "E-Mon": .25, "E5": .20}
def to_m(w):
    s = w.copy(); s.index = pd.to_datetime(s.index).to_period("M"); return s.groupby(level=0).sum()
SRC = {"v7": v7_monthly(), "v4": v4_monthly(), "E5": e5_monthly(), "E-Mon": to_m(emon_weekly_gated("riskoff"))}
M = pd.DataFrame({k: SRC[k] for k in WEIGHTS}).dropna()
D = sum(z(M[k]) * WEIGHTS[k] for k in WEIGHTS).dropna()
src_tag = "Drive" if drive_got else "Yahoo(fixed-window)"
print(f"分散案D[RG3] (30:25:25:20) n={len(D)}ヶ月 span={D.index.min()}..{D.index.max()} / 入力={src_tag}")
print("  (サニティ: Drive入力なら E-Mon raw net≈+67%。上の E-Mon スリーブ表示で確認)")

out = {"data_source": src_tag, "weights": WEIGHTS, "challenge": {}, "funded": {},
       "params": dict(account_usd=ACCOUNT_USD, split=SPLIT, fx=FX)}

def median_month(s, sc, t): return simulate(s, sc, t, n_paths=12000)["med_month"]
def scale_for_median(s, want, t, lo=0.05, hi=12.0):
    for _ in range(28):
        mid = (lo + hi) / 2
        if median_month(s, mid, t) <= want: hi = mid
        else: lo = mid
    return hi
def ann_return(s, sc): v = s.values * sc; return float(np.prod(1 + v) ** (12 / len(v)) - 1)
def annual_fail(s, sc, n=20000, ml=0.10, seed=5):
    P = block_paths(s.values * sc, n, 12, seed=seed); f = 0
    for i in range(n):
        eq = np.cumprod(1 + P[i]); ddv = (eq - np.maximum.accumulate(eq)) / np.maximum.accumulate(eq)
        if ddv.min() <= -ml: f += 1
    return f / n

S3 = scale_for_median(D, 3, 0.08)
print("\n【1】審査 Phase1(+8%/-10%):")
print(f"{'サイズ':24s}{'p95年DD':>9}{'通過%':>8}{'失格%':>8}{'中央月':>7}")
for tag, S in [("median-3 標準", S3), ("median-3 x1.25", S3 * 1.25), ("median-3 x1.5 高リスク", S3 * 1.5)]:
    r = simulate(D, S, 0.08, n_paths=30000); dd = p95_annual_maxdd(D, S)
    out["challenge"][tag] = dict(mult_vs_std=round(S / S3, 2), p95DD=round(dd, 0),
                                 pass_pct=r["pass_pct"], fail_pct=r["fail_pct"], med=r["med_month"])
    print(f"{tag:24s}{dd:>9.0f}{r['pass_pct']:>8}{r['fail_pct']:>8}{r['med_month']:>7}")

S10 = find_scale(D, -10.0)
print(f"\n【2】資金化後 収益  口座=${ACCOUNT_USD:,.0f} / 分配{SPLIT*100:.0f}% / ¥{FX}/$:")
print(f"{'サイズ':16s}{'年率%':>8}{'年失格%':>9}{'手取り/月':>13}{'手取り/年':>13}{'期待手取り/年':>15}")
for tag, S in [("保守 -6%", find_scale(D, -6.0)), ("中庸 -8%", find_scale(D, -8.0)),
               ("攻め -10%", S10), ("高リスク x1.5", S10 * 1.5), ("超攻め x2.0", S10 * 2.0)]:
    ar = ann_return(D, S); af = annual_fail(D, S)
    gy = ACCOUNT_USD * ar * SPLIT * FX; gm = gy / 12; ey = gy * (1 - af)
    out["funded"][tag] = dict(ann_return_pct=round(ar*100,1), annual_fail_pct=round(af*100,1),
                              net_month_jpy=round(gm), net_year_jpy=round(gy), exp_year_jpy=round(ey))
    print(f"{tag:16s}{ar*100:>8.1f}{af*100:>9.1f}{('¥'+format(round(gm),',')):>13}{('¥'+format(round(gy),',')):>13}{('¥'+format(round(ey),',')):>15}")

import os, json as _json
os.makedirs("results", exist_ok=True)
_json.dump(out, open("results/portfolio_D_deploy.json", "w"), ensure_ascii=False, indent=2, default=str)
print("\n保存: results/portfolio_D_deploy.json")
print("注: 審査=median-3攻め(口座喪失前提)。資金化後は別サイズ(生存優先)。¥は近似(Drive+デモで確定)。$50k≈半額。")

try:
    save_result
except NameError:
    pass
else:
    import glob
    save_result("portfolio_D_deploy", metrics=out, inputs=sorted(glob.glob(os.path.join(DATA,"*_d.csv"))), seed=5)
